# Market Basket Analysis and Recommendation Strategy

**Input:** `../Data/Processed/transactions_master.parquet`  
**Main outputs:**
- Frequent itemsets and association rules from FP-Growth.
- Item-to-item recommendation candidates from cosine similarity.
- Cross-sell / up-sell gift bundle strategy for retention campaign.

## 1. Environment Setup

Install the required packages before running the notebook:

```bash
pip install pandas numpy pyarrow fastparquet mlxtend scikit-learn matplotlib seaborn
```

Recommended package roles:
- `pandas`, `numpy`: data loading and aggregation.
- `mlxtend`: FP-Growth and association rules.
- `scikit-learn`: cosine similarity for recommendation.
- `matplotlib`, `seaborn`: quick visual checks and reporting charts.
- `pyarrow`, `fastparquet`: parquet file support.

In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    from mlxtend.frequent_patterns import fpgrowth, association_rules
    from sklearn.metrics.pairwise import cosine_similarity
except ImportError as exc:
    raise ImportError(
        'Missing package. Run: pip install pandas numpy pyarrow fastparquet mlxtend scikit-learn matplotlib seaborn'
    ) from exc

PROJECT_ROOT = Path('..').resolve()
DATA_PATH = PROJECT_ROOT / 'Data' / 'Processed' / 'transactions_master.parquet'
OUTPUT_DIR = PROJECT_ROOT / 'Data' / 'Intermediate' / 'market_basket'
# REPORT_PATH = PROJECT_ROOT / 'reports' / 'internal_briefs' / 'market_basket_summary.md'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

DATA_PATH, OUTPUT_DIR
# REPORT_PATH

(WindowsPath('E:/DSEB/K6_DSEB/DataDr-Mar/DDM-Churn-Project/Data/Processed/transactions_master.parquet'),
 WindowsPath('E:/DSEB/K6_DSEB/DataDr-Mar/DDM-Churn-Project/Data/Intermediate/market_basket'))

## 2. Load and Validate Input Data

The transaction master table is line-item level: one row represents one product in one basket. For MBA and item similarity, the basket identifier is `BASKET_ID`, while item labels can be built from `COMMODITY_DESC`, `DEPARTMENT`, `BRAND`, or `PRODUCT_ID`.


- Use `COMMODITY_DESC` for the main MBA model because it is interpretable and avoids an overly sparse product-level matrix.
- Use `PRODUCT_ID` only for a secondary detailed model if runtime allows.
- Keep `BRAND` and `DEPARTMENT` for strategy interpretation, not as the main basket item.


In [3]:
transactions = pd.read_parquet(DATA_PATH)

print(transactions.shape)
display(transactions.head())
display(transactions.dtypes)

(2548770, 15)


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC,DEPARTMENT,BRAND,COMMODITY_DESC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0,PRODUCE,Private,POTATOES
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0,PRODUCE,National,ONIONS
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0,PRODUCE,Private,VEGETABLES - ALL OTHERS
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0,PRODUCE,National,TROPICAL FRUIT
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0,PRODUCE,Private,ORGANICS FRUIT & VEGETABLES


household_key            str
BASKET_ID                str
DAY                    int16
PRODUCT_ID               str
QUANTITY               int32
SALES_VALUE          float32
STORE_ID                 str
RETAIL_DISC          float32
TRANS_TIME             int16
WEEK_NO                 int8
COUPON_DISC          float32
COUPON_MATCH_DISC    float32
DEPARTMENT               str
BRAND                    str
COMMODITY_DESC           str
dtype: object

In [4]:
required_cols = {
    'household_key', 'BASKET_ID', 'PRODUCT_ID', 'QUANTITY', 'SALES_VALUE',
    'DEPARTMENT', 'BRAND', 'COMMODITY_DESC'
}
missing_cols = required_cols - set(transactions.columns)
assert not missing_cols, f'Missing required columns: {missing_cols}'

summary = {
    'rows': len(transactions),
    'households': transactions['household_key'].nunique(),
    'baskets': transactions['BASKET_ID'].nunique(),
    'products': transactions['PRODUCT_ID'].nunique(),
    'commodities': transactions['COMMODITY_DESC'].nunique(),
    'departments': transactions['DEPARTMENT'].nunique(),
}
summary

{'rows': 2548770,
 'households': 2500,
 'baskets': 251124,
 'products': 91803,
 'commodities': 306,
 'departments': 41}

Because the transaction table contains 91,803 unique products but only 306 commodity categories, COMMODITY_DESC was selected as the main item granularity. This reduces sparsity while keeping the resulting association rules interpretable.

In [5]:
# Check: basket size 
basket_size = (
    transactions
    .groupby('BASKET_ID')['COMMODITY_DESC']
    .nunique()
    .describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
)

basket_size

count    251124.000000
mean          7.470485
std           8.208944
min           1.000000
50%           4.000000
75%          10.000000
90%          19.000000
95%          25.000000
99%          38.000000
max          85.000000
Name: COMMODITY_DESC, dtype: float64

In [6]:
single_item_basket_rate = (
    transactions.groupby('BASKET_ID')['COMMODITY_DESC'].nunique().eq(1).mean()
)

single_item_basket_rate

np.float64(0.1795726414042465)

In [7]:
# Check: Basket matrix shape
basket_matrix = (
    transactions
    .drop_duplicates(['BASKET_ID', 'COMMODITY_DESC'])
    .assign(value=1)
    .pivot_table(
        index='BASKET_ID',
        columns='COMMODITY_DESC',
        values='value',
        fill_value=0
    )
    .astype(bool)
)

basket_matrix.shape

(251124, 306)

## 3. Basket Preparation

FP-Growth requires a one-hot basket matrix where rows are baskets and columns are items. To control memory usage, this notebook filters to items with sufficient basket support before pivoting.

Baseline configuration:
- Item unit: `COMMODITY_DESC`.
- Basket unit: `BASKET_ID`.
- Minimum basket count: adjust depending on matrix size and rule quality.
- Binary encoding: item appears in basket or not, independent of quantity.

In [8]:
ITEM_COL = 'COMMODITY_DESC'
BASKET_COL = 'BASKET_ID'

basket_items = (
    transactions[[BASKET_COL, ITEM_COL]]
    .dropna(subset=[BASKET_COL, ITEM_COL])
    .drop_duplicates([BASKET_COL, ITEM_COL])
)

item_basket_counts = basket_items.groupby(ITEM_COL)[BASKET_COL].nunique().sort_values(ascending=False)


In [9]:
total_baskets = transactions[BASKET_COL].nunique()

thresholds = [50, 100, 150, 300, 500, 1000]

threshold_check = []

for t in thresholds:
    n_items = (item_basket_counts >= t).sum()
    min_support = t / total_baskets
    
    threshold_check.append({
        'MIN_BASKET_COUNT': t,
        'eligible_items': n_items,
        'min_support': min_support
    })

threshold_check = pd.DataFrame(threshold_check)
display(threshold_check)

,MIN_BASKET_COUNT,eligible_items,min_support
0,50,286,0.000199
1,100,275,0.000398
2,150,267,0.000597
3,300,246,0.001195
4,500,229,0.001991
5,1000,204,0.003982


We selected `MIN_BASKET_COUNT = 300` because it provides a better balance between coverage and rule stability. Compared with the 150 threshold, it only removes 21 additional commodity groups, while doubling the minimum basket support from approximately 0.06% to 0.12%. This reduces rare-item noise without substantially narrowing the product space.

In [10]:
MIN_BASKET_COUNT = 300   

eligible_items = item_basket_counts[item_basket_counts >= MIN_BASKET_COUNT].index

basket_items_filtered = basket_items[basket_items[ITEM_COL].isin(eligible_items)]

print('Eligible items:', len(eligible_items))
print('Filtered rows:', len(basket_items_filtered))
display(item_basket_counts.head(20))

Eligible items: 246
Filtered rows: 1869691


COMMODITY_DESC
SOFT DRINKS                  71545
FLUID MILK PRODUCTS          69198
BAKED BREAD/BUNS/ROLLS       60263
CHEESE                       46856
BAG SNACKS                   41975
BEEF                         36695
TROPICAL FRUIT               32466
EGGS                         27866
REFRGRATD JUICES/DRNKS       25920
COLD CEREAL                  25128
LUNCHMEAT                    23697
SOUP                         23576
CANDY - CHECKLANE            23239
FROZEN PIZZA                 23142
FRZN MEAT/MEAT DINNERS       22746
CRACKERS/MISC BKD FD         22702
VEGETABLES - SHELF STABLE    22588
CANDY - PACKAGED             21503
CANNED JUICES                20165
ICE CREAM/MILK/SHERBTS       20059
Name: BASKET_ID, dtype: int64

In [11]:
basket_matrix = (
    basket_items_filtered
    .assign(value=True)
    .pivot_table(index=BASKET_COL, columns=ITEM_COL, values='value', aggfunc='max', fill_value=False)
    .astype(bool)
)

print(basket_matrix.shape)
basket_matrix.head()

(250583, 246)


COMMODITY_DESC,ADULT INCONTINENCE,AIR CARE,ANALGESICS,ANTACIDS,APPAREL,APPLES,AUDIO/VIDEO PRODUCTS,AUTOMOTIVE PRODUCTS,BABY FOODS,BABY HBC,BACON,BAG SNACKS,BAKED BREAD/BUNS/ROLLS,BAKED SWEET GOODS,BAKING,BAKING MIXES,BAKING NEEDS,BATH,BATH TISSUES,BATTERIES,BEANS - CANNED GLASS & MW,BEEF,BEERS/ALES,BERRIES,BEVERAGE,BLEACH,BOOKSTORE,BREAD,BREAKFAST SAUSAGE/SANDWICHES,BREAKFAST SWEETS,BROCCOLI/CAULIFLOWER,BROOMS AND MOPS,BUTTER,CAKES,CANDLES/ACCESSORIES,CANDY - CHECKLANE,CANDY - PACKAGED,CANNED JUICES,CANNED MILK,CARROTS,...,SERVICE BEVERAGE,SHAVING CARE PRODUCTS,SHORTENING/OIL,SINUS AND ALLERGY,SMOKED MEATS,SNACK NUTS,SNACKS,SNKS/CKYS/CRKR/CNDY,SOAP - LIQUID & BAR,SOFT DRINKS,SOUP,SPICES & EXTRACTS,SPRING/SUMMER SEASONAL,SQUASH,STATIONERY & SCHOOL SUPPLIES,STONE FRUIT,SUGARS/SWEETNERS,SUNTAN,SUSHI,SWEET GOODS & SNACKS,SYRUPS/TOPPINGS,TEAS,TICKETS,TOBACCO OTHER,TOMATOES,TOYS AND GAMES,TROPICAL FRUIT,TURKEY,UNKNOWN,VALENTINE,VALUE ADDED FRUIT,VALUE ADDED VEGETABLES,VEGETABLES - ALL OTHERS,VEGETABLES - SHELF STABLE,VEGETABLES SALAD,VITAMINS,WAREHOUSE SNACKS,WATER,WATER - CARBONATED/FLVRD DRINK,YOGURT
BASKET_ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
26984851472,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False
26984851516,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
26984896261,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
26984905972,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
26984945254,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


## 4. Market Basket Analysis with FP-Growth

Association rule interpretation:
- `support`: share of analysis baskets containing the full itemset.
- `confidence`: probability of buying the consequent when the antecedent is present.
- `lift`: rule strength compared with random co-occurrence; values above 1 indicate positive association.

For retention strategy, prioritize rules with high lift and enough basket coverage. Very rare rules can look strong statistically but are hard to operationalize in a campaign.

### 4.1 Generate Frequent Itemsets and Association Rules

This step runs FP-Growth on the filtered commodity-level basket matrix. `basket_count` is calculated from the number of baskets actually used in `basket_matrix`, not from the unfiltered transaction table, so the count is consistent with FP-Growth support.

In [12]:
ANALYSIS_BASKETS = basket_matrix.shape[0]
ANALYSIS_ITEMS = basket_matrix.shape[1]

MIN_SUPPORT = MIN_BASKET_COUNT / ANALYSIS_BASKETS
MIN_CONFIDENCE = 0.05
MIN_LIFT = 1.20
MAX_ITEMSET_LEN = 3

print(f'Analysis baskets: {ANALYSIS_BASKETS:,}')
print(f'Analysis items: {ANALYSIS_ITEMS:,}')
print(f'Min support: {MIN_SUPPORT:.6f}')
print(f'Min confidence: {MIN_CONFIDENCE:.2f}')
print(f'Min lift: {MIN_LIFT:.2f}')

Analysis baskets: 250,583
Analysis items: 246
Min support: 0.001197
Min confidence: 0.05
Min lift: 1.20


In [13]:
frequent_itemsets = fpgrowth(
    basket_matrix,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_ITEMSET_LEN,
)

frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)

if frequent_itemsets.empty or frequent_itemsets['length'].max() < 2:
    rules = pd.DataFrame()
else:
    rules = association_rules(
        frequent_itemsets,
        metric='confidence',
        min_threshold=MIN_CONFIDENCE,
    )

if not rules.empty:
    rules = rules[rules['lift'] >= MIN_LIFT].copy()
    rules['basket_count'] = (rules['support'] * ANALYSIS_BASKETS).round().astype(int)
    rules['antecedent_len'] = rules['antecedents'].apply(len)
    rules['consequent_len'] = rules['consequents'].apply(len)
    rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ' + '.join(sorted(x)))
    rules['consequents_str'] = rules['consequents'].apply(lambda x: ' + '.join(sorted(x)))
    rules = rules.sort_values(['lift', 'confidence', 'support'], ascending=False).reset_index(drop=True)

print('Frequent itemsets:', len(frequent_itemsets))
print('Rules after base lift filter:', len(rules))

if not rules.empty:
    display(
        rules[
            ['antecedents_str', 'consequents_str', 'basket_count', 'support', 'confidence', 'lift']
        ].head(30)
    )

Frequent itemsets: 94904
Rules after base lift filter: 351084


,antecedents_str,consequents_str,basket_count,support,confidence,lift
0,FROZEN,BAKED BREAD/BUNS/ROLLS + REFRIGERATED,350,0.001397,0.226391,28.181648
1,BAKED BREAD/BUNS/ROLLS + REFRIGERATED,FROZEN,350,0.001397,0.173870,28.181648
2,REFRIGERATED + TROPICAL FRUIT,FROZEN,304,0.001213,0.169171,27.420011
3,FROZEN,REFRIGERATED + TROPICAL FRUIT,304,0.001213,0.196636,27.420011
4,REFRIGERATED,BAKED BREAD/BUNS/ROLLS + FROZEN,350,0.001397,0.067738,23.188364
5,BAKED BREAD/BUNS/ROLLS + FROZEN,REFRIGERATED,350,0.001397,0.478142,23.188364
6,FROZEN + TROPICAL FRUIT,REFRIGERATED,304,0.001213,0.468413,22.716532
7,REFRIGERATED,FROZEN + TROPICAL FRUIT,304,0.001213,0.058835,22.716532
8,SEAFOOD - MISC,SEAFOOD-FRESH,337,0.001345,0.296916,20.256514
9,SEAFOOD-FRESH,SEAFOOD - MISC,337,0.001345,0.091751,20.256514


In [14]:
if rules.empty:
    rule_metric_summary = pd.DataFrame()
else:
    rule_metric_summary = rules[
        ['basket_count', 'support', 'confidence', 'lift']
    ].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])


rule_metric_summary

,basket_count,support,confidence,lift
count,351084.000000,351084.000000,351084.000000,351084.000000
mean,814.288600,0.003250,0.230487,3.937668
std,773.502121,0.003087,0.161008,1.393231
min,300.000000,0.001197,0.050000,1.200221
50%,574.000000,0.002291,0.191007,3.727379
75%,927.000000,0.003699,0.308867,4.619890
90%,1505.000000,0.006006,0.464340,5.664722
95%,2065.000000,0.008241,0.585235,6.429287
99%,3909.000000,0.015600,0.705957,8.326894
max,30971.000000,0.123596,0.848321,28.181648


### 4.2 Select Actionable Rule Candidates

The candidate filter keeps rules that are easier to turn into business actions:
- at most two antecedent items,
- exactly one recommended consequent,
- sufficient basket coverage,
- high confidence,
- and strong lift.

These thresholds are intentionally stricter than the FP-Growth base thresholds because this output is meant for campaign planning, not only rule discovery.

In [15]:
CANDIDATE_MIN_BASKET_COUNT = 1500
CANDIDATE_MIN_CONFIDENCE = 0.40
CANDIDATE_MIN_LIFT = 5.00
MAX_ANTECEDENT_LEN = 2

if rules.empty:
    candidate_rules = rules.copy()
else:
    candidate_rules = rules[
        (rules['antecedent_len'] <= MAX_ANTECEDENT_LEN) &
        (rules['consequent_len'] == 1) &
        (rules['basket_count'] >= CANDIDATE_MIN_BASKET_COUNT) &
        (rules['confidence'] >= CANDIDATE_MIN_CONFIDENCE) &
        (rules['lift'] >= CANDIDATE_MIN_LIFT)
    ].copy()

candidate_rules = candidate_rules.sort_values(
    ['lift', 'confidence', 'basket_count'],
    ascending=False,
).reset_index(drop=True)

print('Candidate rules:', len(candidate_rules))

if not candidate_rules.empty:
    display(
        candidate_rules[
            ['antecedents_str', 'consequents_str', 'basket_count', 'support', 'confidence', 'lift']
        ].head(20)
    )

Candidate rules: 406


,antecedents_str,consequents_str,basket_count,support,confidence,lift
0,DRY NOODLES/PASTA + FROZEN BREAD/DOUGH,PASTA SAUCE,1555,0.006206,0.697309,14.906491
1,DINNER MXS:DRY + DRY NOODLES/PASTA,PASTA SAUCE,1923,0.007674,0.600187,12.830296
2,DRY NOODLES/PASTA + FROZEN PIZZA,PASTA SAUCE,1953,0.007794,0.592896,12.674433
3,LAUNDRY ADDITIVES,LAUNDRY DETERGENTS,1888,0.007534,0.424365,12.565126
4,DRY NOODLES/PASTA + LUNCHMEAT,PASTA SAUCE,1881,0.007506,0.576640,12.326924
5,DRY NOODLES/PASTA + MEAT - SHELF STABLE,PASTA SAUCE,1704,0.006800,0.574899,12.289700
6,DRY BN/VEG/POTATO/RICE + DRY NOODLES/PASTA,PASTA SAUCE,1825,0.007283,0.567652,12.134776
7,DRY NOODLES/PASTA + FRZN MEAT/MEAT DINNERS,PASTA SAUCE,1639,0.006541,0.563424,12.044399
8,COLD CEREAL + DRY NOODLES/PASTA,PASTA SAUCE,2320,0.009258,0.562834,12.031780
9,BEEF + DRY NOODLES/PASTA,PASTA SAUCE,3686,0.014710,0.557134,11.909944


In [16]:
if candidate_rules.empty:
    candidate_diagnostics = pd.DataFrame()
else:
    print('Top consequents:')
    display(candidate_rules['consequents_str'].value_counts().head(20).to_frame('n_rules'))

    print('Top antecedents:')
    display(candidate_rules['antecedents_str'].value_counts().head(20).to_frame('n_rules'))

    candidate_diagnostics = candidate_rules[
        ['support', 'confidence', 'lift', 'basket_count']
    ].describe()

candidate_diagnostics

Top consequents:


,n_rules
consequents_str,
VEGETABLES - SHELF STABLE,146
SOUP,61
PASTA SAUCE,29
DRY NOODLES/PASTA,27
COLD CEREAL,20
DELI MEATS,14
DINNER MXS:DRY,14
CHEESES,13
FROZEN PIZZA,13


Top antecedents:


,n_rules
antecedents_str,
PASTA SAUCE + SOUP,4
DINNER MXS:DRY + FRZN MEAT/MEAT DINNERS,4
CONVENIENT BRKFST/WHLSM SNACKS + VEGETABLES - SHELF STABLE,4
DINNER MXS:DRY + DRY NOODLES/PASTA,3
DRY NOODLES/PASTA + FROZEN PIZZA,3
DRY NOODLES/PASTA + MEAT - SHELF STABLE,3
DRY BN/VEG/POTATO/RICE + DRY NOODLES/PASTA,3
COLD CEREAL + DRY NOODLES/PASTA,3
CRACKERS/MISC BKD FD + DRY NOODLES/PASTA,3


,support,confidence,lift,basket_count
count,406.000000,406.000000,406.000000,406.000000
mean,0.008483,0.508358,6.848348,2125.716749
std,0.003127,0.069391,2.159558,783.622510
min,0.005994,0.400511,5.000857,1502.000000
25%,0.006602,0.462831,5.242778,1654.250000
50%,0.007459,0.493493,5.739000,1869.000000
75%,0.008933,0.538484,7.751058,2238.500000
max,0.028462,0.761881,14.906491,7132.000000


### 4.3 Business Scoring and Consequent Diversity

Candidate rules can still be concentrated around a few consequents. To avoid a campaign list dominated by repeated offers, this step adds business-aware scoring and limits the number of rules retained per consequent.

The score combines lift, confidence, and basket count. Rules marked as trivial are not deleted because they can still help recommendations or layout decisions, but their score is discounted for campaign selection.

In [17]:
def is_trivial_rule(row):
    ant = row['antecedents_str'].upper()
    cons = row['consequents_str'].upper()

    trivial_patterns = [
        ('DRY NOODLES/PASTA', 'PASTA SAUCE'),
        ('PASTA SAUCE', 'DRY NOODLES/PASTA'),
        ('CHEESES', 'DELI MEATS'),
        ('DELI MEATS', 'CHEESES'),
        ('LAUNDRY ADDITIVES', 'LAUNDRY DETERGENTS'),
        ('LAUNDRY DETERGENTS', 'LAUNDRY ADDITIVES'),
        ('ONIONS', 'VEGETABLES - ALL OTHERS'),
        ('PEPPERS', 'VEGETABLES - ALL OTHERS'),
    ]

    return int(any(a in ant and c in cons for a, c in trivial_patterns))

candidate_rules = candidate_rules.copy()

if candidate_rules.empty:
    limited_rules = candidate_rules.copy()
else:
    candidate_rules['trivial_flag'] = candidate_rules.apply(is_trivial_rule, axis=1)

    score_cols = ['lift', 'confidence', 'basket_count']
    norm_cols = ['lift_norm', 'confidence_norm', 'basket_count_norm']

    for source_col, norm_col in zip(score_cols, norm_cols):
        col_min = candidate_rules[source_col].min()
        col_max = candidate_rules[source_col].max()
        if col_max == col_min:
            candidate_rules[norm_col] = 1.0
        else:
            candidate_rules[norm_col] = (
                (candidate_rules[source_col] - col_min) / (col_max - col_min)
            )

    candidate_rules['business_score'] = (
        0.40 * candidate_rules['lift_norm'] +
        0.35 * candidate_rules['confidence_norm'] +
        0.25 * candidate_rules['basket_count_norm']
    )

    candidate_rules.loc[candidate_rules['trivial_flag'] == 1, 'business_score'] *= 0.4

    limited_rules = (
        candidate_rules
        .sort_values(['business_score', 'lift', 'confidence', 'basket_count'], ascending=False)
        .groupby('consequents_str', group_keys=False)
        .head(3)
        .reset_index(drop=True)
    )

print('Candidate rules before diversity cap:', len(candidate_rules))
print('Rules after limiting repeated consequents:', len(limited_rules))

if not limited_rules.empty:
    display(
        limited_rules[
            ['antecedents_str', 'consequents_str', 'basket_count', 'confidence', 'lift', 'trivial_flag', 'business_score']
        ].head(30)
    )

Candidate rules before diversity cap: 406
Rules after limiting repeated consequents: 61


,antecedents_str,consequents_str,basket_count,confidence,lift,trivial_flag,business_score
0,BEANS - CANNED GLASS & MW + DRY NOODLES/PASTA,VEGETABLES - SHELF STABLE,1638,0.711246,7.890305,0,0.423677
1,BEANS - CANNED GLASS & MW + CRACKERS/MISC BKD FD,VEGETABLES - SHELF STABLE,2106,0.685324,7.602731,0,0.407739
2,BEANS - CANNED GLASS & MW + BEEF,VEGETABLES - SHELF STABLE,3060,0.648305,7.192059,0,0.397664
3,PASTA SAUCE + PORK,BEEF,1507,0.761881,5.202735,0,0.358374
4,MEAT - SHELF STABLE + PORK,BEEF,1615,0.738792,5.045069,0,0.334441
5,DRY NOODLES/PASTA + PORK,BEEF,1616,0.733545,5.009237,0,0.327957
6,DRY BN/VEG/POTATO/RICE + MEAT - SHELF STABLE,DINNER MXS:DRY,1575,0.509544,9.598784,0,0.294513
7,BAKED BREAD/BUNS/ROLLS + CHEESES,DELI MEATS,4851,0.728378,10.664908,1,0.277994
8,DRY NOODLES/PASTA + FROZEN BREAD/DOUGH,PASTA SAUCE,1555,0.697309,14.906491,1,0.275926
9,CHEESES,DELI MEATS,7132,0.630035,9.224970,1,0.257151


### 4.4 Business Handling Groups

The limited rules are split into three handling groups:

**Group A - Campaign-ready rules:** non-trivial rules with high relative business score. These are the best candidates for coupon targeting, cross-sell tests, and retention campaign handoff.

**Group B - Recommendation / layout rules:** trivial but still useful rules. These should be used for frequently-bought-together modules, bundle display, or store layout rather than direct discounting.

**Group C - Manual review rules:** non-trivial rules with lower relative score. These remain available for business review but are not selected automatically for the pilot campaign.



In [18]:
rules_biz = limited_rules.copy()

if rules_biz.empty:
    campaign_rules = rules_biz.copy()
    recommendation_rules = rules_biz.copy()
    manual_review_rules = rules_biz.copy()
else:
    rules_biz['trivial_flag'] = rules_biz['trivial_flag'].astype(int)
    rules_biz = rules_biz.sort_values('business_score', ascending=False).reset_index(drop=True)

    score_cutoff = rules_biz['business_score'].quantile(0.60)
    rules_biz['high_business_score'] = rules_biz['business_score'] >= score_cutoff
    rules_biz['non_trivial'] = rules_biz['trivial_flag'] == 0
    rules_biz['trivial_but_useful'] = rules_biz['trivial_flag'] == 1

    conditions = [
        rules_biz['non_trivial'] & rules_biz['high_business_score'],
        rules_biz['trivial_but_useful'],
    ]
    choices = ['A_campaign_ready', 'B_recommendation_layout']

    rules_biz['rule_group'] = np.select(
        conditions,
        choices,
        default='C_manual_review',
    )

    campaign_rules = rules_biz[rules_biz['rule_group'] == 'A_campaign_ready'].copy()
    recommendation_rules = rules_biz[rules_biz['rule_group'] == 'B_recommendation_layout'].copy()
    manual_review_rules = rules_biz[rules_biz['rule_group'] == 'C_manual_review'].copy()

display_cols = [
    'antecedents_str',
    'consequents_str',
    'basket_count',
    'confidence',
    'lift',
    'trivial_flag',
    'business_score',
    'rule_group',
]

if rules_biz.empty:
    print('No limited rules available for business grouping.')
else:
    print('Group summary')
    display(
        rules_biz
        .groupby('rule_group')
        .agg(
            n_rules=('rule_group', 'size'),
            avg_basket_count=('basket_count', 'mean'),
            avg_confidence=('confidence', 'mean'),
            avg_lift=('lift', 'mean'),
            avg_business_score=('business_score', 'mean'),
        )
        .reset_index()
    )

    print('Campaign-ready rules')
    display(campaign_rules[display_cols].sort_values('business_score', ascending=False))

    print('Recommendation / layout rules')
    display(recommendation_rules[display_cols].sort_values('business_score', ascending=False))

    print('Manual review rules')
    display(manual_review_rules[display_cols].sort_values('business_score', ascending=False))

Group summary


,rule_group,n_rules,avg_basket_count,avg_confidence,avg_lift,avg_business_score
0,A_campaign_ready,17,2197.352941,0.576585,6.802048,0.274146
1,B_recommendation_layout,12,4100.833333,0.576577,11.016468,0.211538
2,C_manual_review,32,1994.875000,0.441465,6.430469,0.119281


Campaign-ready rules


,antecedents_str,consequents_str,basket_count,confidence,lift,trivial_flag,business_score,rule_group
0,BEANS - CANNED GLASS & MW + DRY NOODLES/PASTA,VEGETABLES - SHELF STABLE,1638,0.711246,7.890305,0,0.423677,A_campaign_ready
1,BEANS - CANNED GLASS & MW + CRACKERS/MISC BKD FD,VEGETABLES - SHELF STABLE,2106,0.685324,7.602731,0,0.407739,A_campaign_ready
2,BEANS - CANNED GLASS & MW + BEEF,VEGETABLES - SHELF STABLE,3060,0.648305,7.192059,0,0.397664,A_campaign_ready
3,PASTA SAUCE + PORK,BEEF,1507,0.761881,5.202735,0,0.358374,A_campaign_ready
4,MEAT - SHELF STABLE + PORK,BEEF,1615,0.738792,5.045069,0,0.334441,A_campaign_ready
5,DRY NOODLES/PASTA + PORK,BEEF,1616,0.733545,5.009237,0,0.327957,A_campaign_ready
6,DRY BN/VEG/POTATO/RICE + MEAT - SHELF STABLE,DINNER MXS:DRY,1575,0.509544,9.598784,0,0.294513,A_campaign_ready
10,BEEF + FROZEN BREAD/DOUGH,PASTA SAUCE,1518,0.460978,9.854394,0,0.255266,A_campaign_ready
11,CONVENIENT BRKFST/WHLSM SNACKS + FLUID MILK PRODUCTS,COLD CEREAL,4276,0.521336,5.198902,0,0.248201,A_campaign_ready
15,FROZEN PIZZA + MEAT - SHELF STABLE,DINNER MXS:DRY,1941,0.457567,8.619648,0,0.220886,A_campaign_ready


Recommendation / layout rules


,antecedents_str,consequents_str,basket_count,confidence,lift,trivial_flag,business_score,rule_group
7,BAKED BREAD/BUNS/ROLLS + CHEESES,DELI MEATS,4851,0.728378,10.664908,1,0.277994,B_recommendation_layout
8,DRY NOODLES/PASTA + FROZEN BREAD/DOUGH,PASTA SAUCE,1555,0.697309,14.906491,1,0.275926,B_recommendation_layout
9,CHEESES,DELI MEATS,7132,0.630035,9.224970,1,0.257151,B_recommendation_layout
12,BAG SNACKS + CHEESES,DELI MEATS,3127,0.725859,10.628018,1,0.245800,B_recommendation_layout
13,BEEF + PASTA SAUCE,DRY NOODLES/PASTA,3686,0.623794,11.151618,1,0.224645,B_recommendation_layout
14,PASTA SAUCE,DRY NOODLES/PASTA,6423,0.547944,9.795638,1,0.221972,B_recommendation_layout
17,DINNER MXS:DRY + DRY NOODLES/PASTA,PASTA SAUCE,1923,0.600187,12.830296,1,0.211300,B_recommendation_layout
18,CHEESE + PASTA SAUCE,DRY NOODLES/PASTA,3944,0.595680,10.649025,1,0.210218,B_recommendation_layout
27,DELI MEATS,CHEESES,7132,0.416735,9.224970,1,0.174515,B_recommendation_layout
34,BAKED BREAD/BUNS/ROLLS + DELI MEATS,CHEESES,4851,0.446850,9.891602,1,0.156435,B_recommendation_layout


Manual review rules


,antecedents_str,consequents_str,basket_count,confidence,lift,trivial_flag,business_score,rule_group
25,DINNER MXS:DRY + FRZN MEAT/MEAT DINNERS,FROZEN PIZZA,1938,0.536842,5.812959,0,0.184196,C_manual_review
26,PEPPERS-ALL,ONIONS,3477,0.422993,6.815070,0,0.182734,C_manual_review
28,CHEESE + PEPPERS-ALL,ONIONS,1846,0.462309,7.448517,0,0.173968,C_manual_review
29,FRZN MEAT/MEAT DINNERS + MEAT - SHELF STABLE,FROZEN PIZZA,1927,0.523215,5.665402,0,0.164550,C_manual_review
30,BEEF + PEPPERS-ALL,ONIONS,1575,0.463918,7.474432,0,0.164539,C_manual_review
31,DINNER MXS:DRY + SOUP,MEAT - SHELF STABLE,2108,0.425429,7.726697,0,0.161116,C_manual_review
32,BEEF + VEGETABLES SALAD,TOMATOES,1819,0.450025,7.391751,0,0.158579,C_manual_review
33,CRACKERS/MISC BKD FD + DINNER MXS:DRY,MEAT - SHELF STABLE,1815,0.430707,7.822562,0,0.157089,C_manual_review
35,BAKED BREAD/BUNS/ROLLS + PAPER TOWELS,BATH TISSUES,2176,0.416938,7.672023,0,0.153704,C_manual_review
36,FLUID MILK PRODUCTS + PAPER TOWELS,BATH TISSUES,2158,0.414203,7.621702,0,0.148224,C_manual_review


In [19]:
MAX_RULES_PER_CONSEQUENT_B = 3
MAX_RULES_PER_CONSEQUENT_C = 1
N_MANUAL_REVIEW_RULES = 15

if recommendation_rules.empty:
    final_recommendation_rules = recommendation_rules.copy()
else:
    final_recommendation_rules = (
        recommendation_rules
        .sort_values(
            ['consequents_str', 'business_score', 'confidence', 'lift'],
            ascending=[True, False, False, False],
        )
        .groupby('consequents_str')
        .head(MAX_RULES_PER_CONSEQUENT_B)
        .sort_values('business_score', ascending=False)
        .reset_index(drop=True)
    )

if manual_review_rules.empty:
    manual_review_shortlist = manual_review_rules.copy()
else:
    manual_review_shortlist = (
        manual_review_rules
        .sort_values(
            ['consequents_str', 'business_score', 'basket_count'],
            ascending=[True, False, False],
        )
        .groupby('consequents_str')
        .head(MAX_RULES_PER_CONSEQUENT_C)
        .sort_values('business_score', ascending=False)
        .head(N_MANUAL_REVIEW_RULES)
        .reset_index(drop=True)
    )

print('Final recommendation/layout rules')
if final_recommendation_rules.empty:
    print('No recommendation/layout rules selected.')
else:
    display(final_recommendation_rules[display_cols])

print('Manual review shortlist')
if manual_review_shortlist.empty:
    print('No manual review rules selected.')
else:
    display(manual_review_shortlist[display_cols])

Final recommendation/layout rules


,antecedents_str,consequents_str,basket_count,confidence,lift,trivial_flag,business_score,rule_group
0,BAKED BREAD/BUNS/ROLLS + CHEESES,DELI MEATS,4851,0.728378,10.664908,1,0.277994,B_recommendation_layout
1,DRY NOODLES/PASTA + FROZEN BREAD/DOUGH,PASTA SAUCE,1555,0.697309,14.906491,1,0.275926,B_recommendation_layout
2,CHEESES,DELI MEATS,7132,0.630035,9.224970,1,0.257151,B_recommendation_layout
3,BAG SNACKS + CHEESES,DELI MEATS,3127,0.725859,10.628018,1,0.245800,B_recommendation_layout
4,BEEF + PASTA SAUCE,DRY NOODLES/PASTA,3686,0.623794,11.151618,1,0.224645,B_recommendation_layout
5,PASTA SAUCE,DRY NOODLES/PASTA,6423,0.547944,9.795638,1,0.221972,B_recommendation_layout
6,DINNER MXS:DRY + DRY NOODLES/PASTA,PASTA SAUCE,1923,0.600187,12.830296,1,0.211300,B_recommendation_layout
7,CHEESE + PASTA SAUCE,DRY NOODLES/PASTA,3944,0.595680,10.649025,1,0.210218,B_recommendation_layout
8,DELI MEATS,CHEESES,7132,0.416735,9.224970,1,0.174515,B_recommendation_layout
9,BAKED BREAD/BUNS/ROLLS + DELI MEATS,CHEESES,4851,0.446850,9.891602,1,0.156435,B_recommendation_layout


Manual review shortlist


,antecedents_str,consequents_str,basket_count,confidence,lift,trivial_flag,business_score,rule_group
0,DINNER MXS:DRY + FRZN MEAT/MEAT DINNERS,FROZEN PIZZA,1938,0.536842,5.812959,0,0.184196,C_manual_review
1,PEPPERS-ALL,ONIONS,3477,0.422993,6.815070,0,0.182734,C_manual_review
2,DINNER MXS:DRY + SOUP,MEAT - SHELF STABLE,2108,0.425429,7.726697,0,0.161116,C_manual_review
3,BEEF + VEGETABLES SALAD,TOMATOES,1819,0.450025,7.391751,0,0.158579,C_manual_review
4,BAKED BREAD/BUNS/ROLLS + PAPER TOWELS,BATH TISSUES,2176,0.416938,7.672023,0,0.153704,C_manual_review
5,COOKIES/CONES + SOUP,CRACKERS/MISC BKD FD,2083,0.497255,5.488661,0,0.139197,C_manual_review
6,CHEESE + MISC. DAIRY,HISPANIC,1702,0.429040,7.541917,0,0.139123,C_manual_review
7,CARROTS,VEGETABLES - ALL OTHERS,3395,0.404167,6.196225,0,0.135870,C_manual_review
8,DINNER MXS:DRY + VEGETABLES - SHELF STABLE,DRY BN/VEG/POTATO/RICE,2068,0.401553,7.587848,0,0.130608,C_manual_review
9,CHEESE + HISPANIC,MILK BY-PRODUCTS,3347,0.420425,5.349137,0,0.115278,C_manual_review


### 4.5 Final Campaign Rules and Export

The campaign table first locks the original top 10 rules. After that top-10 list is fixed, the manually listed `recipe_trap` rules are removed immediately before saving the CSV. Removed rules are not replaced, so two manual removals from the top 10 produce an 8-row final campaign file.

The CSV saved at the end of this section is the handoff file for campaign planning.

In [20]:
MAX_RULES_PER_CONSEQUENT_A = 2
N_INITIAL_CAMPAIGN_RULES = 10

manual_rule_antecedent = 'DINNER MXS:DRY + FRZN MEAT/MEAT DINNERS'
manual_rule_consequent = 'FROZEN PIZZA'

MANUAL_DEMOTE_RULES = [
    ('PASTA SAUCE + PORK', 'BEEF', 'recipe_trap'),
    ('BEEF + FROZEN BREAD/DOUGH', 'PASTA SAUCE', 'recipe_trap'),
]

def norm_rule_text(x):
    return ' '.join(str(x).strip().upper().split())

manual_demote_df = pd.DataFrame(
    MANUAL_DEMOTE_RULES,
    columns=['antecedents_str', 'consequents_str', 'manual_risk_type'],
)
manual_demote_df['_rule_key'] = (
    manual_demote_df['antecedents_str'].map(norm_rule_text)
    + ' -> '
    + manual_demote_df['consequents_str'].map(norm_rule_text)
)
manual_demote_keys = set(manual_demote_df['_rule_key'])

if manual_review_rules.empty:
    forced_rule = manual_review_rules.copy()
else:
    forced_rule = manual_review_rules[
        (manual_review_rules['antecedents_str'] == manual_rule_antecedent) &
        (manual_review_rules['consequents_str'] == manual_rule_consequent)
    ].copy()

if not forced_rule.empty:
    forced_rule['rule_group'] = 'A_campaign_ready_manual_add'

# Step 1: lock the original top 10 campaign list first.
campaign_pool = pd.concat(
    [campaign_rules, forced_rule],
    ignore_index=True,
).drop_duplicates(
    subset=['antecedents_str', 'consequents_str'],
    keep='first',
)

if campaign_pool.empty:
    initial_top10_campaign_rules = campaign_pool.copy()
else:
    campaign_capped = (
        campaign_pool
        .sort_values(
            ['consequents_str', 'business_score', 'basket_count'],
            ascending=[True, False, False],
        )
        .groupby('consequents_str')
        .head(MAX_RULES_PER_CONSEQUENT_A)
        .reset_index(drop=True)
    )

    if forced_rule.empty:
        initial_top10_campaign_rules = (
            campaign_capped
            .sort_values('business_score', ascending=False)
            .head(N_INITIAL_CAMPAIGN_RULES)
            .reset_index(drop=True)
        )
    else:
        base_top = (
            campaign_capped
            .merge(
                forced_rule[['antecedents_str', 'consequents_str']],
                on=['antecedents_str', 'consequents_str'],
                how='left',
                indicator=True,
            )
        )

        base_top = (
            base_top[base_top['_merge'] == 'left_only']
            .drop(columns='_merge')
            .sort_values('business_score', ascending=False)
            .head(max(N_INITIAL_CAMPAIGN_RULES - len(forced_rule), 0))
        )

        initial_top10_campaign_rules = pd.concat(
            [base_top, forced_rule],
            ignore_index=True,
        ).sort_values(
            'business_score',
            ascending=False,
        ).head(N_INITIAL_CAMPAIGN_RULES).reset_index(drop=True)

# Step 2: only after top 10 is locked, manually remove recipe-trap rules.
initial_top10_campaign_rules['_rule_key'] = (
    initial_top10_campaign_rules['antecedents_str'].map(norm_rule_text)
    + ' -> '
    + initial_top10_campaign_rules['consequents_str'].map(norm_rule_text)
)

manual_removed_campaign_rules = (
    initial_top10_campaign_rules[
        initial_top10_campaign_rules['_rule_key'].isin(manual_demote_keys)
    ]
    .merge(
        manual_demote_df[['_rule_key', 'manual_risk_type']],
        on='_rule_key',
        how='left',
    )
    .drop(columns='_rule_key')
)

final_campaign_rules = (
    initial_top10_campaign_rules[
        ~initial_top10_campaign_rules['_rule_key'].isin(manual_demote_keys)
    ]
    .drop(columns='_rule_key')
    .reset_index(drop=True)
)

final_campaign_export_cols = [
    'antecedents_str',
    'consequents_str',
    'basket_count',
    'support',
    'confidence',
    'lift',
    'trivial_flag',
    'business_score',
    'rule_group',
]

final_campaign_rules = final_campaign_rules[final_campaign_export_cols].copy()
final_campaign_rules.insert(0, 'campaign_rule_rank', range(1, len(final_campaign_rules) + 1))
final_campaign_rules['recommended_campaign_use'] = 'Retention cross-sell coupon / next-best-offer pilot'

FINAL_CAMPAIGN_RULES_PATH = OUTPUT_DIR / 'final_campaign_rules.csv'
final_campaign_rules.to_csv(FINAL_CAMPAIGN_RULES_PATH, index=False, encoding='utf-8-sig')

print('Initial campaign rules before manual removal:', len(initial_top10_campaign_rules))
print('Manual removed campaign rules:', len(manual_removed_campaign_rules))
if not manual_removed_campaign_rules.empty:
    display(manual_removed_campaign_rules[['antecedents_str', 'consequents_str', 'manual_risk_type']])

print('Final campaign rules saved:', len(final_campaign_rules))
print('Saved to:', FINAL_CAMPAIGN_RULES_PATH)
display(final_campaign_rules)

Initial campaign rules before manual removal: 10
Manual removed campaign rules: 2


,antecedents_str,consequents_str,manual_risk_type
0,PASTA SAUCE + PORK,BEEF,recipe_trap
1,BEEF + FROZEN BREAD/DOUGH,PASTA SAUCE,recipe_trap


Final campaign rules saved: 8
Saved to: E:\DSEB\K6_DSEB\DataDr-Mar\DDM-Churn-Project\Data\Intermediate\market_basket\final_campaign_rules.csv


,campaign_rule_rank,antecedents_str,consequents_str,basket_count,support,confidence,lift,trivial_flag,business_score,rule_group,recommended_campaign_use
0,1,BEANS - CANNED GLASS & MW + DRY NOODLES/PASTA,VEGETABLES - SHELF STABLE,1638,0.006537,0.711246,7.890305,0,0.423677,A_campaign_ready,Retention cross-sell coupon / next-best-offer pilot
1,2,BEANS - CANNED GLASS & MW + CRACKERS/MISC BKD FD,VEGETABLES - SHELF STABLE,2106,0.008404,0.685324,7.602731,0,0.407739,A_campaign_ready,Retention cross-sell coupon / next-best-offer pilot
2,3,MEAT - SHELF STABLE + PORK,BEEF,1615,0.006445,0.738792,5.045069,0,0.334441,A_campaign_ready,Retention cross-sell coupon / next-best-offer pilot
3,4,DRY BN/VEG/POTATO/RICE + MEAT - SHELF STABLE,DINNER MXS:DRY,1575,0.006285,0.509544,9.598784,0,0.294513,A_campaign_ready,Retention cross-sell coupon / next-best-offer pilot
4,5,CONVENIENT BRKFST/WHLSM SNACKS + FLUID MILK PRODUCTS,COLD CEREAL,4276,0.017064,0.521336,5.198902,0,0.248201,A_campaign_ready,Retention cross-sell coupon / next-best-offer pilot
5,6,FROZEN PIZZA + MEAT - SHELF STABLE,DINNER MXS:DRY,1941,0.007746,0.457567,8.619648,0,0.220886,A_campaign_ready,Retention cross-sell coupon / next-best-offer pilot
6,7,BAKED BREAD/BUNS/ROLLS + CONVENIENT BRKFST/WHLSM SNACKS,COLD CEREAL,3929,0.015679,0.502365,5.009720,0,0.206779,A_campaign_ready,Retention cross-sell coupon / next-best-offer pilot
7,8,DINNER MXS:DRY + FRZN MEAT/MEAT DINNERS,FROZEN PIZZA,1938,0.007734,0.536842,5.812959,0,0.184196,A_campaign_ready_manual_add,Retention cross-sell coupon / next-best-offer pilot


## 5. Item-to-Item Recommendation with Cosine Similarity

The recommendation model uses the same filtered basket matrix. Each commodity is represented by its basket-presence vector, and cosine similarity measures whether two commodities tend to appear in similar baskets.

This complements FP-Growth in three ways:
- It produces a recommendation list for every eligible anchor item, not only item pairs that pass association-rule thresholds.
- It is symmetric, so it is useful for item-neighborhood discovery and display recommendations.
- It can support retention-gift shortlists by connecting a customer's known purchase categories to relevant coupon candidates.


### 5.1 Build the Item Similarity Matrix

The input matrix is transposed so rows are commodities and columns are baskets. A co-basket threshold is kept alongside cosine similarity to avoid recommendations that look similar only because of very small overlap.


In [21]:
TOP_N_RECOMMENDATIONS = 10
MIN_REC_COSINE = 0.12
MIN_REC_CO_BASKET_COUNT = 300

item_matrix = basket_matrix.T.astype(np.int8)
item_similarity = cosine_similarity(item_matrix)
np.fill_diagonal(item_similarity, 0)

# Use int32 to avoid overflow when counting co-baskets across many baskets.
item_matrix_values = item_matrix.to_numpy(dtype=np.int32, copy=False)
co_basket_count_matrix = item_matrix_values @ item_matrix_values.T

item_labels = basket_matrix.columns.to_numpy()
item_support_lookup = item_basket_counts.reindex(item_labels).astype(int)

filtered_similarity_mask = (
    (item_similarity >= MIN_REC_COSINE) &
    (co_basket_count_matrix >= MIN_REC_CO_BASKET_COUNT)
)
filtered_item_similarity = item_similarity.copy()
filtered_item_similarity[~filtered_similarity_mask] = 0
np.fill_diagonal(filtered_item_similarity, 0)

print('Item matrix shape:', item_matrix.shape)
print('Similarity matrix shape:', item_similarity.shape)
print('Filtered non-zero similarity pairs:', np.count_nonzero(filtered_item_similarity))


Item matrix shape: (246, 250583)
Similarity matrix shape: (246, 246)
Filtered non-zero similarity pairs: 5352


In [22]:
recommendation_rows = []

for anchor_idx, anchor_item in enumerate(item_labels):
    sorted_candidate_idx = np.argsort(filtered_item_similarity[anchor_idx])[::-1]
    kept_rank = 0

    for candidate_idx in sorted_candidate_idx:
        similarity_score = filtered_item_similarity[anchor_idx, candidate_idx]
        co_basket_count = co_basket_count_matrix[anchor_idx, candidate_idx]

        if similarity_score < MIN_REC_COSINE:
            break
        if co_basket_count < MIN_REC_CO_BASKET_COUNT:
            continue

        kept_rank += 1
        recommended_item = item_labels[candidate_idx]

        recommendation_rows.append({
            'anchor_item': anchor_item,
            'recommended_item': recommended_item,
            'recommendation_rank': kept_rank,
            'cosine_similarity': similarity_score,
            'co_basket_count': co_basket_count,
            'anchor_basket_count': item_support_lookup.loc[anchor_item],
            'recommended_basket_count': item_support_lookup.loc[recommended_item],
        })

        if kept_rank >= TOP_N_RECOMMENDATIONS:
            break

item_recommendations = pd.DataFrame(recommendation_rows)

item_recommendations['recommendation_score'] = (
    item_recommendations['cosine_similarity']
    * np.log1p(item_recommendations['co_basket_count'])
)

item_recommendations = item_recommendations.sort_values(
    ['anchor_item', 'recommendation_rank']
).reset_index(drop=True)

print('Recommendation rows:', len(item_recommendations))
print('Anchor items with recommendations:', item_recommendations['anchor_item'].nunique())

display(item_recommendations.head(30))

ITEM_RECOMMENDATIONS_PATH = OUTPUT_DIR / 'item_recommendations.csv'
item_recommendations.to_csv(ITEM_RECOMMENDATIONS_PATH, index=False, encoding='utf-8-sig')
print('Item-to-item recommendations saved to:', ITEM_RECOMMENDATIONS_PATH)


Recommendation rows: 1158
Anchor items with recommendations: 147


,anchor_item,recommended_item,recommendation_rank,cosine_similarity,co_basket_count,anchor_basket_count,recommended_basket_count,recommendation_score
0,AIR CARE,HOUSEHOLD CLEANG NEEDS,1,0.203294,1078,3808,7384,1.419762
1,AIR CARE,LAUNDRY DETERGENTS,2,0.135285,768,3808,8463,0.898983
2,AIR CARE,LAUNDRY ADDITIVES,3,0.135081,556,3808,4449,0.854060
3,APPLES,TROPICAL FRUIT,1,0.315752,6539,13210,32466,2.774096
4,APPLES,BAKED BREAD/BUNS/ROLLS,2,0.248380,7008,13210,60263,2.199396
5,APPLES,FLUID MILK PRODUCTS,3,0.242540,7333,13210,69198,2.158674
6,APPLES,CITRUS,4,0.234240,2707,13210,10110,1.851426
7,APPLES,CHEESE,5,0.228948,5696,13210,46856,1.979868
8,APPLES,YOGURT,6,0.225579,3439,13210,17594,1.836940
9,APPLES,COLD CEREAL,7,0.211809,3859,13210,25128,1.749208


Item-to-item recommendations saved to: E:\DSEB\K6_DSEB\DataDr-Mar\DDM-Churn-Project\Data\Intermediate\market_basket\item_recommendations.csv


### 5.2 Personalized Voucher Recommendations by Household

`item_recommendations` above is an item-to-item lookup table. The personalized voucher file below converts the filtered similarity matrix into household-level next-best-item recommendations:
- build each household's purchased commodity vector,
- score every candidate commodity by normalized top-k anchor similarity to the household's purchased commodities,
- remove commodities the household has already purchased,
- apply department-level diversity so the Top 3 is not concentrated in one group,
- keep the Top 3 unpurchased commodities per household for personalized voucher targeting.


In [23]:
TOP_N_PERSONALIZED_VOUCHERS = 3
TOP_K_ANCHORS_FOR_SCORE = 5
MIN_PERSONALIZED_SCORE = 0
MAX_RECS_PER_ITEM_GROUP = 1

item_group_lookup = (
    transactions[[ITEM_COL, 'DEPARTMENT']]
    .dropna(subset=[ITEM_COL, 'DEPARTMENT'])
    .groupby(ITEM_COL)['DEPARTMENT']
    .agg(lambda x: x.value_counts().idxmax())
    .reindex(item_labels)
    .fillna('UNKNOWN')
)
item_group_array = item_group_lookup.to_numpy()

household_items = (
    transactions[['household_key', ITEM_COL]]
    .dropna(subset=['household_key', ITEM_COL])
    .drop_duplicates(['household_key', ITEM_COL])
)

household_items_filtered = household_items[household_items[ITEM_COL].isin(item_labels)]

household_item_matrix = (
    household_items_filtered
    .assign(value=True)
    .pivot_table(
        index='household_key',
        columns=ITEM_COL,
        values='value',
        aggfunc='max',
        fill_value=False,
    )
    .reindex(columns=item_labels, fill_value=False)
    .astype(bool)
)

household_item_values = household_item_matrix.to_numpy(dtype=bool, copy=False)
household_keys = household_item_matrix.index.to_numpy()
item_support_array = item_support_lookup.reindex(item_labels).to_numpy(dtype=int)
personalized_rows = []

def select_diverse_top_items(score_vector, candidate_idx, item_groups, top_n, max_per_group):
    sorted_idx = candidate_idx[np.argsort(score_vector[candidate_idx])[::-1]]
    selected_idx = []
    selected_group_counts = {}
    relaxed_items = set()

    for idx in sorted_idx:
        group = item_groups[idx]
        if selected_group_counts.get(group, 0) >= max_per_group:
            continue
        selected_idx.append(idx)
        selected_group_counts[group] = selected_group_counts.get(group, 0) + 1
        if len(selected_idx) >= top_n:
            return selected_idx, relaxed_items

    # If a household has too few valid groups, fill the remaining slots by score.
    for idx in sorted_idx:
        if idx in selected_idx:
            continue
        selected_idx.append(idx)
        relaxed_items.add(idx)
        if len(selected_idx) >= top_n:
            break

    return selected_idx, relaxed_items

for household_pos, household_key in enumerate(household_keys):
    purchased_idx = np.flatnonzero(household_item_values[household_pos])
    if len(purchased_idx) == 0:
        continue

    anchor_candidate_similarity = filtered_item_similarity[purchased_idx, :]
    k_anchor_count = min(TOP_K_ANCHORS_FOR_SCORE, len(purchased_idx))
    top_anchor_similarity = np.sort(anchor_candidate_similarity, axis=0)[-k_anchor_count:, :]
    score_vector = top_anchor_similarity.sum(axis=0) / k_anchor_count

    # Do not recommend commodities that the household has already purchased.
    score_vector = score_vector.astype(float, copy=False)
    score_vector[purchased_idx] = -np.inf

    valid_candidate_idx = np.flatnonzero(score_vector > MIN_PERSONALIZED_SCORE)

    if len(valid_candidate_idx) == 0:
        continue

    top_candidate_idx, diversity_relaxed_idx = select_diverse_top_items(
        score_vector=score_vector,
        candidate_idx=valid_candidate_idx,
        item_groups=item_group_array,
        top_n=TOP_N_PERSONALIZED_VOUCHERS,
        max_per_group=MAX_RECS_PER_ITEM_GROUP,
    )

    for rank, candidate_idx in enumerate(top_candidate_idx, start=1):
        anchor_similarity = filtered_item_similarity[purchased_idx, candidate_idx]
        strongest_anchor_pos = anchor_similarity.argmax()
        strongest_anchor_idx = purchased_idx[strongest_anchor_pos]

        personalized_rows.append({
            'household_key': household_key,
            'voucher_recommendation_rank': rank,
            'recommended_item': item_labels[candidate_idx],
            'recommended_item_group': item_group_array[candidate_idx],
            'predicted_purchase_score': score_vector[candidate_idx],
            'strongest_purchased_anchor_item': item_labels[strongest_anchor_idx],
            'strongest_purchased_anchor_group': item_group_array[strongest_anchor_idx],
            'anchor_item_similarity': anchor_similarity[strongest_anchor_pos],
            'household_eligible_item_count': len(purchased_idx),
            'recommended_item_basket_count': item_support_array[candidate_idx],
            'score_method': f'top_{TOP_K_ANCHORS_FOR_SCORE}_filtered_similarity_mean',
            'diversity_relaxed_flag': int(candidate_idx in diversity_relaxed_idx),
        })

personalized_voucher_recommendations = pd.DataFrame(personalized_rows)

personalized_voucher_recommendations = personalized_voucher_recommendations.sort_values(
    ['household_key', 'voucher_recommendation_rank']
).reset_index(drop=True)

PERSONALIZED_VOUCHER_RECOMMENDATIONS_PATH = OUTPUT_DIR / 'personalized_voucher_recommendations.csv'
personalized_voucher_recommendations.to_csv(
    PERSONALIZED_VOUCHER_RECOMMENDATIONS_PATH,
    index=False,
    encoding='utf-8-sig',
)

print('Households with eligible purchase history:', household_item_matrix.shape[0])
print('Personalized voucher recommendation rows:', len(personalized_voucher_recommendations))
print('Households with recommendations:', personalized_voucher_recommendations['household_key'].nunique())
print('Saved to:', PERSONALIZED_VOUCHER_RECOMMENDATIONS_PATH)

display(personalized_voucher_recommendations.head(30))


Households with eligible purchase history: 2500
Personalized voucher recommendation rows: 7499
Households with recommendations: 2500
Saved to: E:\DSEB\K6_DSEB\DataDr-Mar\DDM-Churn-Project\Data\Intermediate\market_basket\personalized_voucher_recommendations.csv


,household_key,voucher_recommendation_rank,recommended_item,recommended_item_group,predicted_purchase_score,strongest_purchased_anchor_item,strongest_purchased_anchor_group,anchor_item_similarity,household_eligible_item_count,recommended_item_basket_count,score_method,diversity_relaxed_flag
0,1,1,FROZEN PIZZA,GROCERY,0.292994,FRZN MEAT/MEAT DINNERS,GROCERY,0.313383,124,23142,top_5_filtered_similarity_mean,0
1,1,2,CHICKEN,MEAT,0.248231,BEEF,MEAT,0.285648,124,15042,top_5_filtered_similarity_mean,0
2,1,3,BREAKFAST SAUSAGE/SANDWICHES,MEAT-PCKGD,0.241292,EGGS,GROCERY,0.258854,124,10971,top_5_filtered_similarity_mean,0
3,10,1,CHEESE,GROCERY,0.401183,BAKED BREAD/BUNS/ROLLS,GROCERY,0.450089,40,46856,top_5_filtered_similarity_mean,0
4,10,2,TROPICAL FRUIT,PRODUCE,0.317018,FLUID MILK PRODUCTS,GROCERY,0.377251,40,32466,top_5_filtered_similarity_mean,0
5,10,3,HOT DOGS,MEAT-PCKGD,0.241066,BAKED BREAD/BUNS/ROLLS,GROCERY,0.300342,40,10921,top_5_filtered_similarity_mean,0
6,100,1,DRY NOODLES/PASTA,GROCERY,0.334019,PASTA SAUCE,GROCERY,0.501083,96,14017,top_5_filtered_similarity_mean,0
7,100,2,TROPICAL FRUIT,PRODUCE,0.329796,FLUID MILK PRODUCTS,GROCERY,0.377251,96,32466,top_5_filtered_similarity_mean,0
8,100,3,CHEESES,DELI,0.273206,DELI MEATS,DELI,0.512404,96,11320,top_5_filtered_similarity_mean,0
9,1000,1,BEEF,MEAT,0.358179,BAKED BREAD/BUNS/ROLLS,GROCERY,0.399766,128,36695,top_5_filtered_similarity_mean,0


### 5.3 Inspect Personalized Voucher Output

These checks are required before treating the recommender output as campaign-ready. They show whether the same items dominate the whole file, which anchor-to-recommended pairs drive the score, how much household history is available, and a household-level sample for manual review.


In [24]:
if personalized_voucher_recommendations.empty:
    print('No personalized voucher recommendations to inspect.')
else:
    print('Top recommended items across all households')
    top_recommended_items = (
        personalized_voucher_recommendations
        .groupby(['recommended_item', 'recommended_item_group'])
        .agg(
            n_households=('household_key', 'nunique'),
            avg_rank=('voucher_recommendation_rank', 'mean'),
            avg_score=('predicted_purchase_score', 'mean'),
            avg_anchor_similarity=('anchor_item_similarity', 'mean'),
        )
        .reset_index()
        .sort_values(['n_households', 'avg_score'], ascending=False)
    )
    display(top_recommended_items.head(25))

    print('Top strongest anchor -> recommended item pairs')
    top_anchor_recommended_pairs = (
        personalized_voucher_recommendations
        .groupby([
            'strongest_purchased_anchor_item',
            'strongest_purchased_anchor_group',
            'recommended_item',
            'recommended_item_group',
        ])
        .agg(
            n_households=('household_key', 'nunique'),
            avg_score=('predicted_purchase_score', 'mean'),
            avg_anchor_similarity=('anchor_item_similarity', 'mean'),
        )
        .reset_index()
        .sort_values(['n_households', 'avg_anchor_similarity'], ascending=False)
    )
    display(top_anchor_recommended_pairs.head(25))

    print('Distribution by household eligible item count')
    household_history_distribution = (
        personalized_voucher_recommendations
        .drop_duplicates('household_key')
        ['household_eligible_item_count']
        .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
        .to_frame('household_eligible_item_count')
    )
    display(household_history_distribution)

    print('Recommendation count by number of distinct recommended groups per household')
    household_recommended_group_diversity = (
        personalized_voucher_recommendations
        .groupby('household_key')
        .agg(
            n_recommendations=('recommended_item', 'size'),
            n_recommended_groups=('recommended_item_group', 'nunique'),
            diversity_relaxed_count=('diversity_relaxed_flag', 'sum'),
        )
        .reset_index()
    )
    display(
        household_recommended_group_diversity
        .groupby(['n_recommendations', 'n_recommended_groups'])
        .agg(
            n_households=('household_key', 'nunique'),
            households_with_relaxed_diversity=('diversity_relaxed_count', lambda x: (x > 0).sum()),
        )
        .reset_index()
        .sort_values(['n_recommendations', 'n_recommended_groups'], ascending=False)
    )

    print('Sample 20 households')
    sample_households = (
        personalized_voucher_recommendations['household_key']
        .drop_duplicates()
        .sample(n=min(20, personalized_voucher_recommendations['household_key'].nunique()), random_state=42)
    )
    display(
        personalized_voucher_recommendations[
            personalized_voucher_recommendations['household_key'].isin(sample_households)
        ]
        .sort_values(['household_key', 'voucher_recommendation_rank'])
    )


Top recommended items across all households


,recommended_item,recommended_item_group,n_households,avg_rank,avg_score,avg_anchor_similarity
41,DELI MEATS,DELI,570,2.198246,0.281644,0.386534
29,CHEESES,DELI,359,1.983287,0.275122,0.512404
130,TROPICAL FRUIT,PRODUCE,337,1.857567,0.316004,0.372377
83,LUNCHMEAT,MEAT-PCKGD,296,2.250000,0.294381,0.352519
11,BEEF,MEAT,262,1.446565,0.332061,0.388417
93,ONIONS,PRODUCE,233,2.090129,0.275355,0.295501
3,BACON,MEAT-PCKGD,199,2.572864,0.247481,0.277785
129,TOMATOES,PRODUCE,193,2.051813,0.272288,0.306741
104,PORK,MEAT,186,2.435484,0.247589,0.305827
30,CHICKEN,MEAT,150,2.566667,0.246932,0.285648


Top strongest anchor -> recommended item pairs


,strongest_purchased_anchor_item,strongest_purchased_anchor_group,recommended_item,recommended_item_group,n_households,avg_score,avg_anchor_similarity
23,BAKED BREAD/BUNS/ROLLS,GROCERY,DELI MEATS,DELI,400,0.265900,0.338040
89,DELI MEATS,DELI,CHEESES,DELI,359,0.275122,0.512404
119,FLUID MILK PRODUCTS,GROCERY,TROPICAL FRUIT,PRODUCE,300,0.322162,0.377251
31,BAKED BREAD/BUNS/ROLLS,GROCERY,LUNCHMEAT,MEAT-PCKGD,253,0.302744,0.360285
19,BAKED BREAD/BUNS/ROLLS,GROCERY,BEEF,MEAT,208,0.346112,0.399766
100,EGGS,GROCERY,BACON,MEAT-PCKGD,196,0.247611,0.278287
52,BEEF,MEAT,PORK,MEAT,186,0.247589,0.305827
80,CHEESES,DELI,DELI MEATS,DELI,162,0.322522,0.512404
50,BEEF,MEAT,CHICKEN,MEAT,150,0.246932,0.285648
99,DRY NOODLES/PASTA,GROCERY,PASTA SAUCE,GROCERY,147,0.317771,0.501083


Distribution by household eligible item count


,household_eligible_item_count
count,2500.000000
mean,112.237600
std,44.029688
min,2.000000
10%,51.000000
25%,82.000000
50%,115.000000
75%,146.000000
90%,168.000000
95%,180.050000


Recommendation count by number of distinct recommended groups per household


,n_recommendations,n_recommended_groups,n_households,households_with_relaxed_diversity
2,3,3,2491,0
1,3,2,8,8
0,2,2,1,0


Sample 20 households


,household_key,voucher_recommendation_rank,recommended_item,recommended_item_group,predicted_purchase_score,strongest_purchased_anchor_item,strongest_purchased_anchor_group,anchor_item_similarity,household_eligible_item_count,recommended_item_basket_count,score_method,diversity_relaxed_flag
738,122,1,DRY NOODLES/PASTA,GROCERY,0.334019,PASTA SAUCE,GROCERY,0.501083,131,14017,top_5_filtered_similarity_mean,0
739,122,2,TROPICAL FRUIT,PRODUCE,0.325200,FLUID MILK PRODUCTS,GROCERY,0.377251,131,32466,top_5_filtered_similarity_mean,0
740,122,3,HOT DOGS,MEAT-PCKGD,0.245266,BAKED BREAD/BUNS/ROLLS,GROCERY,0.300342,131,10921,top_5_filtered_similarity_mean,0
1206,1360,1,CHEESE,GROCERY,0.371440,BAKED BREAD/BUNS/ROLLS,GROCERY,0.450089,40,46856,top_5_filtered_similarity_mean,0
1207,1360,2,BEEF,MEAT,0.314416,BAKED BREAD/BUNS/ROLLS,GROCERY,0.399766,40,36695,top_5_filtered_similarity_mean,0
1208,1360,3,CHEESES,DELI,0.268409,DELI MEATS,DELI,0.512404,40,11320,top_5_filtered_similarity_mean,0
1413,1422,1,COLD CEREAL,GROCERY,0.335313,FLUID MILK PRODUCTS,GROCERY,0.403607,57,25128,top_5_filtered_similarity_mean,0
1414,1422,2,DELI MEATS,DELI,0.323845,CHEESES,DELI,0.512404,57,17114,top_5_filtered_similarity_mean,0
1415,1422,3,TOMATOES,PRODUCE,0.266733,VEGETABLES - ALL OTHERS,PRODUCE,0.284844,57,15256,top_5_filtered_similarity_mean,0
1491,1446,1,BEEF,MEAT,0.359858,BAKED BREAD/BUNS/ROLLS,GROCERY,0.399766,115,36695,top_5_filtered_similarity_mean,0
